# Étape 2 — GraphSAGE vs XGBoost sur YelpChi (Colab ou local)
Répond à la question de recherche : les relations (GraphSAGE) battent-elles l'approche isolée (XGBoost) ? **Plus besoin de DGL** : YelpChi chargé direct depuis le .mat (scipy). Repo privé → remplace `TON_TOKEN` par un PAT scope `repo`, et clone la branche `impl-pipeline`.

In [ ]:
!pip -q install torch_geometric xgboost scikit-learn

In [ ]:
TOKEN = "TON_TOKEN"  # PAT classic, scope repo
!git clone -b impl-pipeline https://{TOKEN}@github.com/racef-code/pfe-fraude-graph-ml.git
%cd pfe-fraude-graph-ml

In [ ]:
from src.config import TrainConfig, set_seed
from src.data.yelpchi import load_yelpchi
from src.models.graphsage import GraphSAGE
from src.train.train_gnn import train_gnn, predict_scores, class_weights_from_labels
from src.train.baseline_xgb import train_xgb, predict_xgb
from src.eval.metrics import compute_metrics, print_comparison
import numpy as np

set_seed(42)
data = load_yelpchi()
y     = data.y.cpu().numpy()
train = data.train_mask.cpu().numpy()
test  = data.test_mask.cpu().numpy()

# GraphSAGE (relations)
cw  = class_weights_from_labels(data.y[data.train_mask])
gnn = GraphSAGE(data.x.shape[1], 64, 2, dropout=0.5)
gnn = train_gnn(gnn, data, TrainConfig(epochs=200), class_weight=cw)
gnn_scores = predict_scores(gnn, data)

# XGBoost (memes features, SANS graphe)
X = data.x.cpu().numpy()
n_neg, n_pos = (y[train]==0).sum(), (y[train]==1).sum()
xgb = train_xgb(X[train], y[train], scale_pos_weight=n_neg/n_pos)
xgb_scores = predict_xgb(xgb, X)

results = {
    'GraphSAGE': compute_metrics(y[test], gnn_scores[test]),
    'XGBoost':   compute_metrics(y[test], xgb_scores[test]),
}
print_comparison(results)